In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

In [31]:
lidc_dir = Path("Z:/nas-ctm01/datasets/public/Lung_Nodule/test/LIDC_IDRI/annotations")
lndb_dir = Path("Z:/nas-ctm01/datasets/public/Lung_Nodule/test/LNDb/annotations")
nodule_list = sorted([*lidc_dir.glob("*.npz"), *lndb_dir.glob("*.npz")])


In [32]:
print(f"Number of nodules: {len(nodule_list)}")

Number of nodules: 156


In [33]:
sizes = []
spiculation = []
sphericity = []
for nodule in nodule_list:
    info = np.load(nodule)
    sizes.append(info["equiv_diameter_mm"])
    spiculation.append(np.median(info["char_spiculation"]))
    sphericity.append(np.median(info["char_sphericity"]))

    

In [34]:
sizes = np.array(sizes)
spiculation = np.array(spiculation)
sphericity = np.array(sphericity)

print(np.unique(sizes, return_counts=True))
spiculation = np.ceil(spiculation)
sphericity = np.ceil(sphericity)

print(np.unique(spiculation, return_counts=True))
print(np.unique(sphericity, return_counts=True))

(array([1, 2, 3]), array([118,  28,  10]))
(array([1., 2., 3., 4., 5.]), array([115,  28,   9,   2,   2]))
(array([1., 2., 3., 4., 5.]), array([ 2, 13, 36, 57, 48]))


In [12]:
info_lidc = np.load("datasets/LIDC-IDRI-0001_nodule0.npz")
info_lndb = np.load("datasets/LNDb-0001_nodule1.npz")
for key in info_lidc.files:
        print(key, info_lidc[key].shape, info_lidc[key].dtype)
print("\n", "-"*50, "\n")
for key in info_lndb.files:
        print(key, info_lndb[key].shape, info_lndb[key].dtype)


patient_id () <U14
equiv_diameter_mm () int64
bbox (3, 2) int16
mask (9, 52, 44) uint8
spacing (3,) float64
char_subtlety (4,) int64
char_internalStructure (4,) int64
char_calcification (4,) int64
char_sphericity (4,) int64
char_margin (4,) int64
char_lobulation (4,) int64
char_spiculation (4,) int64
char_texture (4,) int64
char_malignancy (4,) int64

 -------------------------------------------------- 

patient_id () <U9
bbox (3, 2) int16
mask (6, 17, 15) uint8
spacing (3,) float64
original_volume_shape (3,) int32
equiv_diameter_mm () int16
char_texture (1,) float32
char_calcification (1,) float32
char_internalstructure (1,) float32
char_lobulation (1,) float32
char_malignancy (1,) float32
char_sphericity (1,) float32
char_spiculation (1,) float32
char_subtlety (1,) float32
char_margin (1,) float32


In [18]:
print(type(info_lndb["char_sphericity"]))

<class 'numpy.ndarray'>


In [26]:
table = pd.DataFrame({
    "equiv_diameter_mm": info_lidc["equiv_diameter_mm"].item(),
    "sphericity": info_lidc["char_sphericity"],
    "spiculation": info_lidc["char_spiculation"],
})

# table.index = range(1, len(table) + 1)
# table.index.name = "annotation"

print(table.to_string())

   equiv_diameter_mm  sphericity  spiculation
0                  3           3            4
1                  3           4            5
2                  3           3            3
3                  3           5            5


## Case 3: annotation filenames versus instance labels

These viewers use the original **NIfTI instance masks**, not the binary NumPy masks or preprocessed volumes. Both cases belong to the training split. The NIfTI arrays are transposed from `(X,Y,Z)` to annotation `(Z,Y,X)` coordinates, **without flipping slices**. Bounding-box upper endpoints are exclusive for these nine annotations.

Checked directly against the source data: all four LIDC annotations match label `X + 1`, and all five LNDb annotations match label `X`, with Dice **1.0** for every annotation. This verifies these two cases, not the entire datasets. The binary LIDC NumPy mask has reversed z order relative to its NPZ annotation coordinates, so it must not be substituted directly here.

Run the next three cells for an independent axial slider per case. Filled regions and their label numbers come from the actual instance mask; dashed boxes and `noduleX` names come from the NPZ files. Use **Jump to annotation**, the slice slider/arrows, and **Zoom to nodules**. Each viewer is also saved as a standalone HTML file under `nodule_checks/`, which can be opened in a browser if the notebook renderer blocks embedded HTML. No ipywidgets installation is required.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import nibabel as nib

# Supports opening this notebook from downstream/ or from the repository root.
downstream_dir = Path.cwd() if (Path.cwd() / "nodule_label_viewer.py").is_file() else Path.cwd() / "downstream"
if str(downstream_dir) not in sys.path:
    sys.path.insert(0, str(downstream_dir))
from nodule_label_viewer import inspect_nodule_labels

source_root = Path("Z:/nas-ctm01/datasets/public")
viewer_dir = downstream_dir / "nodule_checks"


def view_case3(dataset, case_id, label_offset):
    mask_path = source_root / "LungNodule-nifti" / "train" / dataset / "nodulemask" / f"{case_id}.nii.gz"
    annotation_dir = source_root / "Lung_Nodule" / "train" / dataset / "annotations"
    nifti = nib.load(mask_path)
    # Verified against the annotation crops: NO reversal of the z axis.
    mask_zyx = np.asarray(nifti.dataobj).transpose(2, 1, 0)
    document, table = inspect_nodule_labels(
        mask_zyx, annotation_dir, case_id, label_offset,
        html_path=viewer_dir / f"{case_id}.html",
    )
    table.to_csv(viewer_dir / f"{case_id}_overlaps.csv", index=False)
    return document, table


In [ ]:
lidc_case3_html, lidc_case3_table = view_case3("LIDC_IDRI", "LIDC-IDRI-0003", label_offset=1)


In [ ]:
lndb_case3_html, lndb_case3_table = view_case3("LNDb", "LNDb-0003", label_offset=0)
